In [3]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py

--2026-06-23 22:56:51--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-06-23 22:56:52 (93.7 MB/s) - ‘download.py’ saved [1376/1376]

--2026-06-23 22:56:52--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 

In [25]:
from embedder import Embedder
embed = Embedder()
query = "How does approximate nearest neighbor search work?"
v1 = embed.encode(query)
v1[0]

np.float64(-0.020582036807885073)

In [20]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [13]:
for doc in documents:
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        res = v1.dot(embed.encode(doc["content"]))
        print(res)
        break

0.361070280302606


In [21]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
chunks[0].keys()

dict_keys(['start', 'content', 'filename'])

In [22]:

texts = [chunk["filename"] + " " + chunk["content"] for chunk in chunks]

In [23]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
ve = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    ve.extend(batch_vectors)

X = np.array(ve)

  0%|          | 0/6 [00:00<?, ?it/s]

In [24]:
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

chunks[idx]

{'start': 1000,
 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data lives on disk, one 

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [30]:
# query = "What metric do we use to evaluate a search engine?"
query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)

results = vindex.search(query_vector, num_results=5)
for res in results:
    print(res["filename"])


02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [ ]:
query = "How do I store vectors in PostgreSQL?"
from minsearch import Index
def build_index(documents):
    index = Index(
        text_fields=['content'],
        keyword_fields=['filename']
    )
    index.fit(documents)
    return index
def search_files(index, query, filename_filter=None, num_results=5):
    # Basic search
    results = index.search(
        query=query,
        filter_dict={'filename': filename_filter} if filename_filter else {},
        num_results=num_results,
        boost_dict={'content': 1.0}
    )
    return results
index = build_index(chunks)
results = search_files(index, query, num_results=5)
for res in results:
    print(res["filename"])



02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [32]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [33]:
new_query = "How do I give the model access to tools?"
t_results = search_files(index, new_query, num_results=5)
new_query_vector = embed.encode(new_query)
v_results = vindex.search(new_query_vector, num_results=5)
results = rrf([v_results, t_results])

In [ ]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 